# பாடம் 10 - உற்பத்தியில் AI முகவர்கள்  

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கு **உற்பத்தி முறைமைகளை** கற்றுக்கொள்ளப்போகின்றீர்கள். இங்கே நாங்கள் கையாள்கிறோம்:  

- **கண்காணிப்பு** — முகவர் தொடர்புகளில் நேரம் மற்றும் பதிவு சேர்க்கல்  
- **மதிப்பீடு** — ஒத்துழைப்பாளர் முகவரை பயன்படுத்தி பதிலின் தரத்தை மதிப்பீடு செய்வது  
- **செலவுக் கட்டுப்பாடு** — டோக்கன் அப்டிமைசேஷன் மற்றும் மாடல் தேர்வு உத்திகள்  

இந்த சூழல் ஒரு **பயண முகவர்** ஆகும், பயனர்களுக்கு பயண திட்டமிட உதவுகிறதுடன், மேலே கண்காணிப்பு மற்றும் மதிப்பீடு உள்ளது.  


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## உற்பத்தி கவனிக்க வேண்டிய அம்சங்கள்

செயல்முறை மாதிரிகளிலிருந்து AI முகவர்களை உற்பத்திக்கு நகர்த்துவது மூன்று அம்சங்களுக்கு கவனமாக இருக்க வேண்டும்:

1. **காண்பித்தல்** — முகவர் என்ன செய்கிறார், எவ்வளவு நேரம் எடுத்துக்கொள்கிறார், மற்றும் எந்த கருவிகளை அழைக்கிறார் என்பதை நீங்கள் தெரிந்து கொள்ள வேண்டும். தடவல் மற்றும் பதிவு இல்லாமல் உற்பத்தி பிரச்சினைகளைக் கண்டறிவது சிரமமாகும்.

2. **மதிப்பீடு** — தானியங்கி தரமான சரிபார்ப்புகள், முகவரின் பதில்கள் காலத்துடன் துல்லியமாகவும், நிறைவாகவும், உதவிகரமாகவும் இருக்கின்றன என்பதை உறுதிப்படுத்துகின்றன. ஒரு மதிப்பாய்வாளர் முகவர் வரையறுக்கப்பட்ட அளவுகோல்களுக்கு எதிராக பதில்களை மதிப்பிடலாம்.

3. **செலவை மேலாண்மை** — டோக்கன் பயணம் செலவுகளுக்கு நேரடியாக பாதிப்பை ஏற்படுத்துகிறது. மகிழ்ச்சி நுட்பமயமாக்கல், மாதிரி தேர்வு மற்றும் காசே செய்வது போன்ற நடைமுறைகள் செலவுகளை கட்டுப்படுத்த உதவுகின்றன, தரத்தை குறைக்காமல்.


## பார்வையிடக்கூடிய முகவரியை உருவாக்குதல்

நாம் பயண கருவிகளை வரையறுக்கிறோம் மற்றும் முகவரியின் அழைப்பை நேரத்துடன் சுற்றி, விலம்பை கண்காணிக்கலாம். உற்பத்தியில் நீங்கள் OpenTelemetry அல்லது அதே போன்ற கண்காணிப்பு பின்னணி ஒருங்கிணைக்கும்.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## மதிப்பீட்டுக் கட்டமைப்புகள்

ஒரு பொதுவான தயாரிப்பு கட்டமைப்பு, இரண்டாவது முகவரியை **மதிப்பீட்டாளராக** பயன்படுத்துவதாகும். மதிப்பீட்டாளன் முதன்மை முகவரியின் பதிலை முன்னிருப்பு கோட்பாடுகள் போன்று முழுமையான தன்மை, துல்லியம் மற்றும் உதவித்தன்மை ஆகியவற்றுக்கு எதிராக மதிப்பிடுகிறான்.

இது இனிமையாக செய்கிறது:
- பயனர்களுக்கு பதில்கள் செல்லும் முன் தானியங்கி தரநிலை வாசல்கள்
- ஊக்கச்சுவடுகள் அல்லது மாத்ரிகள் மாற்றப்படும் போது பின்தங்கல் கண்டறிதல்
- முகவர் செயல்திறனை காலப்போக்கில் தொடர்ந்த கண்காணித்தல்


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## செலவுக் கட்டுப்பாட்டுக்கான रणனீடுகள்

உற்பத்தி AI முகவரிகளுக்காக செலவுகளை கட்டுப்படுத்துவது மிகவும் முக்கியம். முக்கியமான रणனீடுகள் இங்கே உள்ளன:

| रणனீடு | விளக்கம் |
|---|---|
| **உத்தேசம் மேம்பாடு** | அமைப்பு வழிமுறைகளை சுருக்கமாக வைத்திருங்கள். உள்ளீடு டோக்கன்களை குறைக்க சீரற்ற சூழலை நீக்குங்கள். |
    "| **மாதிரி தேர்வு** | வகைப்படுத்தல் அல்லது எடுத்துக்காட்டு போன்ற எளிய பணிகளுக்காக சிறிய, மலிவான மாதிரிகளைக் (உதாரணம்: GPT-5-mini) பயன்படுத்தவும், மற்றும் சிக்கலான தகவல் பகிர்வுக்கு பெரிய மாதிரிகளை ஒதுக்கவும். |\n",
| **கேச்சிங்** | மீள்மீளும் API அழைப்புகளைத் தவிர்க்க கருவி முடிவுகள் மற்றும் அடிக்கடி கேள்விகள் கைப்படுத்தவும். |
| **டோக்கன் பட்ஜெட்டுகள்** | எதிர்பாராத நீண்ட பதில்களைத் தடுக்கும் வகையில் `max_tokens` வரம்புகளை அமைக்கவும். |
| **பேச்சிங்** | சாத்தியமான இடங்களில் பல பயனர் கேள்விகளை ஒரே API அழைப்பில் குழுவாக்கவும். |

நடைமுறையில், படிநிலை முறை சிறந்தது: எளிதான கோரிக்கைகளை வேகமான, மலிவான மாதிரிக்கு வழிமொழிந்து, சிக்கலான கேள்விகளை மட்டுமே திறமையான மாதிரிக்கு மேம்படுத்தவும்.


## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி என்பதைக் கற்றுக்கொண்டீர்கள்:

1. முகவர்க் காட்சியியல் மற்றும் பதிவு மூலம் முகவர் தொடர்புகளைப் பார்வையிடுதல் சேர்க்கவும், இதன் மூலம் தடயமிடல் மற்றும் நடைமுறை கண்காணிப்பு உருவாக்கப்படுகிறது.
2. மதிப்பீட்டாளர் முகவரைக் கொண்டு தொடர்பான பதில்களை தானாகவே மதிப்பீடு செய்து முழுமை, துல்லியம் மற்றும் உதவித்தன்மையை மதிப்பிடுதல்.
3. நேரம் நியமனத்திற்கான மேம்பாடு, மாடல் தேர்வு, காசே மற்றும் டோக்கன் பட்ஜெட் வழியாக செலவுகளை நிர்வஹித்தல்.

இந்த உற்பத்தித் துறையின் மாதிரிகள் உங்கள் AI முகவர்களை நம்பகமானதும், அளவிடத்தக்கதும், செலவுகுறைப்படும் நிலையானதும் ஆக உறுதி செய்ய உதவுகின்றன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
